# Rate limits, quotas, structured errors, and cursor pagination

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 26 §26.3–§26.4 · type: walkthrough*

**One-line promise:** make the API defend itself — a per-tenant sliding-window limiter that returns `429` + `Retry-After`, daily quotas, a consistent machine-readable error envelope, and cursor pagination — all offline against a shared in-memory store standing in for Redis.

## 🧠 Why this matters

A public API must protect *itself*. Without limits, one buggy client (or one bad actor) can exhaust your capacity, and — because each request here triggers an expensive model call — your budget too. **Rate limiting** caps requests per short window; **quotas** cap usage over a billing period. Both live in *shared* state so they hold across every server instance, and both are keyed *per tenant* so one customer can't starve another.

The other half of "enterprise-grade" is *predictability*: structured errors a client can branch on (not prose to string-match), and pagination so a list endpoint never dumps an unbounded result. Pick these conventions once and every consumer benefits forever.

## Objectives & prereqs

**By the end you can:**
- Implement a sliding-window limiter keyed by API key + user + **tenant**, backed by a shared store, returning `429 Too Many Requests` with a correct `Retry-After`.
- Have a client *honor* `Retry-After` and back off — the contract both sides rely on.
- Distinguish a short-window **rate limit** from a per-day **quota**, and enforce both.
- Return a consistent **error envelope** via one exception handler.
- Page a list endpoint with an opaque **cursor**, and explain why offset breaks on changing data.

**Prereqs:** notebook **26-01** (auth, tenant claim). No API key, no Redis, no network — a shared in-memory store stands in for Redis and the clock is faked for determinism.

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# stdlib only (+ python-dotenv & fastapi from requirements.txt). No network, no Redis.
import os
import json
import time
import base64
import random
import pathlib
from collections import deque, defaultdict

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(26)  # deterministic synthetic data below

# A shared store stands in for Redis. In production this is a real Redis (or fakeredis in
# tests) so limits hold ACROSS instances; the interface (incr/expiring keys) is the same.
# We also fake the clock so the sliding window is fully deterministic in CI.
class FakeClock:
    def __init__(self, t=1_000_000.0):
        self.t = t
    def now(self):
        return self.t
    def advance(self, seconds):
        self.t += seconds
        return self.t

clock = FakeClock()
print(f"MOCK mode: {MOCK}  | shared store = in-memory (Redis stand-in); clock faked")

## 1 · A sliding-window rate limiter, per tenant (§26.3, 🔧 Build)

Key choice: a **sliding window** counts the requests in the last `window` seconds, so it doesn't suffer the burst-at-the-boundary problem of fixed windows. We store request timestamps per key in the shared store; the key is `tenant:user:api_key` so limits are scoped *per tenant* (Ch 28). When the count hits the cap we return how long until the oldest request ages out — that number becomes `Retry-After`.

In [ ]:
class SlidingWindowLimiter:
    """Per-key sliding window over a shared store. Returns (allowed, retry_after).
    Backed here by an in-memory dict of deques; in prod, a Redis sorted-set per key."""
    def __init__(self, limit: int, window_s: float, clock: FakeClock):
        self.limit = limit
        self.window_s = window_s
        self.clock = clock
        self.hits = defaultdict(deque)  # key -> deque[timestamp]

    def check(self, key: str):
        now = self.clock.now()
        dq = self.hits[key]
        # drop timestamps older than the window (they no longer count)
        while dq and dq[0] <= now - self.window_s:
            dq.popleft()
        if len(dq) >= self.limit:
            # retry_after = when the oldest in-window hit will expire
            retry_after = self.window_s - (now - dq[0])
            return False, max(1, int(retry_after + 0.999))  # ceil, at least 1s
        dq.append(now)
        return True, 0

def rate_key(tenant: str, user: str, api_key: str) -> str:
    return f"{tenant}:{user}:{api_key}"

# 3 requests / 10s, just to make the limit easy to hit in the demo.
limiter = SlidingWindowLimiter(limit=3, window_s=10.0, clock=clock)
print("limiter ready: 3 requests / 10s, keyed by tenant:user:api_key")

🔮 **Predict.** With a limit of **3 per 10s**, tenant `acme` sends **5** requests at the same instant (the clock doesn't advance). How many get `allowed=True`, and what `Retry-After` does the *first rejected* one report? Decide before running.

In [ ]:
key = rate_key("acme", "u_alice", "ak_live_123")
results = []
for i in range(5):
    allowed, retry_after = limiter.check(key)
    results.append((i + 1, allowed, retry_after))
for n, allowed, ra in results:
    verdict = "ALLOW" if allowed else f"429 (Retry-After: {ra}s)"
    print(f"  request {n}: {verdict}")

**What you just saw.** The first 3 pass; the 4th and 5th are rejected with `Retry-After: 10` (nothing has aged out yet, so the full window remains). A *different* tenant has its own key and its own fresh budget — one customer can't consume another's.

## 2 · The client side of the contract: honor `Retry-After` (🔧 Build)

A `429` is only useful if clients respect it. A well-behaved client reads `Retry-After`, waits, then retries — instead of hammering and making the overload worse. We simulate that loop against the limiter using the faked clock (no real sleeping), so you can watch the back-off succeed once the window slides.

In [ ]:
def well_behaved_client(limiter, key, max_tries=4):
    """Try; on 429, advance the (faked) clock by Retry-After and retry."""
    for attempt in range(1, max_tries + 1):
        allowed, retry_after = limiter.check(key)
        if allowed:
            return f"succeeded on attempt {attempt}"
        print(f"  attempt {attempt}: 429, backing off {retry_after}s (honoring Retry-After)")
        limiter.clock.advance(retry_after)  # real client would time.sleep(retry_after)
    return "gave up after backoff budget"

# The key from section 1 is already at its limit; a polite client backs off and gets in.
print(well_behaved_client(limiter, key))

**What you just saw.** After waiting the advertised `Retry-After`, the oldest hits aged out of the window and the request succeeded. This is the entire social contract of rate limiting: the server says *how long*, the client *waits that long*. A client that ignores `Retry-After` and retries immediately just collects more `429`s.

## 3 · Rate limit ≠ quota (§26.3)

Two different controls, often confused:

- **Rate limit** — smooths *bursts*: "≤ N requests per few seconds." Resets continuously.
- **Quota** — caps *total volume per plan*: "≤ 10,000 requests per day on the Pro plan." Resets on a calendar boundary; exceeding it usually means *upgrade*, not *wait a moment*.

Quotas are naturally **per tenant** (it's the customer's plan). Here's a simple per-tenant daily counter alongside the limiter.

In [ ]:
class DailyQuota:
    """Per-tenant daily request counter in the shared store. Resets each UTC day."""
    def __init__(self, plan_limits: dict, clock: FakeClock):
        self.plan_limits = plan_limits        # plan -> requests/day
        self.clock = clock
        self.used = defaultdict(int)          # (tenant, day) -> count

    def _day(self):
        return int(self.clock.now() // 86400)

    def charge(self, tenant: str, plan: str):
        cap = self.plan_limits[plan]
        slot = (tenant, self._day())
        if self.used[slot] >= cap:
            return False, {"used": self.used[slot], "limit": cap}
        self.used[slot] += 1
        return True, {"used": self.used[slot], "limit": cap}

quota = DailyQuota({"free": 2, "pro": 10_000}, clock)
# Simulate a free-plan tenant making 3 calls against a daily cap of 2.
for i in range(3):
    ok, info = quota.charge("acme", plan="free")
    state = "OK" if ok else "QUOTA EXCEEDED (upgrade plan)"
    print(f"  call {i + 1}: {state}  used={info['used']}/{info['limit']}")

**What you just saw.** The third call is refused — not because of a *burst*, but because the *daily* allotment is spent. The right client response differs too: back off for a rate limit, upgrade (or wait until tomorrow) for a quota. Same `429`/`402` family, different remedy — which is exactly why your error body must say *which* it is.

## 4 · One structured error envelope (§26.4, 🔧 Build)

Clients should branch on a stable **code**, never on your prose. The book's envelope:

```json
{ "error": { "code": "rate_limited", "message": "Too many requests.", "retry_after": 30 } }
```

Wire it once as a FastAPI exception handler so *every* error — yours and the framework's — comes out in the same shape. Now we mount the limiter as real middleware-ish dependency on a route and watch the `429` carry both the header and the envelope.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header, Query
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

app = FastAPI(title="Agent API", version="1.0.0")

class APIError(HTTPException):
    """Carries a stable machine code + optional retry_after into the envelope."""
    def __init__(self, status_code: int, code: str, message: str, retry_after: int | None = None):
        super().__init__(status_code=status_code, detail=message)
        self.code = code
        self.retry_after = retry_after

@app.exception_handler(APIError)
def api_error_handler(request, exc: APIError):
    body = {"error": {"code": exc.code, "message": exc.detail}}
    headers = {}
    if exc.retry_after is not None:
        body["error"]["retry_after"] = exc.retry_after
        headers["Retry-After"] = str(exc.retry_after)
    return JSONResponse(status_code=exc.status_code, content=body, headers=headers)

# Fresh limiter for the endpoint demo (2 req / 60s) on its own clock.
ep_clock = FakeClock()
ep_limiter = SlidingWindowLimiter(limit=2, window_s=60.0, clock=ep_clock)

def enforce_rate_limit(
    x_tenant: str = Header(default="acme"),
    x_user: str = Header(default="u_alice"),
    x_api_key: str = Header(default="ak_live_123"),
):
    allowed, retry_after = ep_limiter.check(rate_key(x_tenant, x_user, x_api_key))
    if not allowed:
        raise APIError(429, "rate_limited", "Too many requests.", retry_after=retry_after)
    return {"tenant": x_tenant, "user": x_user}

@app.get("/v1/ping")
def ping(ctx: dict = Depends(enforce_rate_limit)):
    return {"ok": True, **ctx}

client = TestClient(app)
for i in range(3):
    r = client.get("/v1/ping")
    print(f"  request {i + 1}: HTTP {r.status_code}  Retry-After={r.headers.get('Retry-After')}  body={r.json()}")

**What you just saw.** The third call returns `429`, a `Retry-After` header, **and** the envelope `{"error": {"code": "rate_limited", ...}}`. A client reads `error.code`, sees `rate_limited`, and runs its back-off path — no string-matching, no guesswork. Every error in the service now shares this shape.

## 5 · Cursor pagination — never return an unbounded list (§26.4, 🔧 Build)

⚠️ **Pitfall: the unbounded list.** A `GET /messages` that returns *everything* is a latency bomb and a memory bomb waiting for the dataset to grow. Always paginate. We use a **cursor** (an opaque token encoding "start after id X") rather than `?offset=`, because offset *skips by position* — when rows are inserted or deleted between pages, offset silently repeats or drops items. A cursor pins to a stable key, so it's correct on a changing dataset.

In [ ]:
# Generate 200 synthetic messages (newest first) -- committed-size data made in-cell.
MESSAGES = [
    {"id": 1000 - i, "tenant": "acme", "text": f"message #{1000 - i}"}
    for i in range(200)
]  # ids 1000..801, descending

def encode_cursor(last_id: int) -> str:
    return base64.urlsafe_b64encode(json.dumps({"after": last_id}).encode()).decode()

def decode_cursor(cursor: str) -> int:
    return json.loads(base64.urlsafe_b64decode(cursor.encode()))["after"]

@app.get("/v1/messages")
def list_messages(limit: int = Query(default=50, le=100), cursor: str | None = None):
    """Cursor pagination: page by `id < after`, newest first. limit is CAPPED at 100."""
    after = decode_cursor(cursor) if cursor else None
    rows = MESSAGES if after is None else [m for m in MESSAGES if m["id"] < after]
    page = rows[:limit]
    next_cursor = encode_cursor(page[-1]["id"]) if len(rows) > limit and page else None
    return {"data": page, "next_cursor": next_cursor}

# Walk the first two pages of 50.
p1 = client.get("/v1/messages?limit=50").json()
print("page 1:", len(p1["data"]), "rows, ids", p1["data"][0]["id"], "->", p1["data"][-1]["id"], "| next?", bool(p1["next_cursor"]))
p2 = client.get(f"/v1/messages?limit=50&cursor={p1['next_cursor']}").json()
print("page 2:", len(p2["data"]), "rows, ids", p2["data"][0]["id"], "->", p2["data"][-1]["id"], "| next?", bool(p2['next_cursor']))
# The cap holds even if a client asks for more:
capped = client.get("/v1/messages?limit=999")
print("limit=999 ->", "HTTP", capped.status_code, "(422: limit must be <= 100, never unbounded)")

**What you just saw.** Page 1 returns ids 1000→951, page 2 continues 950→901 via the cursor, and `limit=999` is *rejected* (`422`) because the schema caps it at 100. There is no way to ask this endpoint for an unbounded list — the pitfall is closed by construction.

## 🎯 Senior lens

The expensive mistake here is *inconsistency*. Pick your conventions **once** and apply them everywhere: cursor (not offset) pagination, the single `{"error": {"code", ...}}` envelope, `snake_case` fields, ISO-8601 **UTC** timestamps, and `Retry-After` on every `429`. Every inconsistency is a tax each client pays forever — extra docs to read, extra branches to write, extra integration bugs. 

Two more senior notes. First, put the limiter's state in a *shared* store (Redis) from day one, even with a single instance — the day you scale out, an in-process counter silently lets through N× your limit. Second, decide your limiter's *failure mode*: if Redis is down, do you *fail open* (serve, risk overload) or *fail closed* (reject, risk an outage)? There's no universal answer, but there must be a *decision*, not an accident.

## Recap

- A **sliding-window** limiter keyed by `tenant:user:api_key` returns `429` + `Retry-After`; the window slides so bursts at the boundary don't slip through.
- The contract is two-sided: the server says *how long*, a well-behaved client **honors** `Retry-After`.
- **Rate limit** (bursts, resets continuously) ≠ **quota** (per-plan volume, resets daily) — different remedies: back off vs upgrade.
- One **error envelope** (`{"error": {"code", "message", "retry_after"}}`) via a single handler; clients branch on `code`, not prose.
- **Cursor pagination** with a capped `limit` makes an unbounded list impossible and stays correct on changing data where offset doesn't.
- Pick conventions once; **shared state** for limits; choose fail-open vs fail-closed deliberately.

## Exercises

Predict the result before running each.

1. **Window slide.** After hitting the limit in section 1, `clock.advance(5)` then check again — still `429`? Now advance another `6`s and check. Explain the `Retry-After` you'd expect at each step.
2. **Token bucket vs sliding window.** Re-implement the limiter as a token bucket (capacity + refill rate). Which one allows a short burst above the average rate, and why might you *want* that for a bursty client?
3. **Quota header.** Add `X-RateLimit-Remaining` and `X-RateLimit-Reset` headers to the `/v1/ping` response. What should `Reset` be relative to — the window or the wall clock?
4. **Offset breaks.** Build an offset-paginated version of `/v1/messages`, fetch page 1, *insert* a new newest message, then fetch page 2 by offset. Show the duplicate/skip. Why is the cursor immune?

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


## Next

- ⬅️ **Previous:** [`26-01-authn-authz-oauth2-jwt-rbac.ipynb`](./26-01-authn-authz-oauth2-jwt-rbac.ipynb).
- ➡️ **Next notebook:** [`26-03-webhooks-and-openapi-contracts.ipynb`](./26-03-webhooks-and-openapi-contracts.ipynb) — notify other systems reliably (signed, retried, idempotent webhooks) and gate API changes with a contract test against OpenAPI.
- 📓 **Book:** §26.3 (rate limiting, quotas, multi-tenancy) and §26.4 (pagination, filtering, errors).
- 🧱 **Template:** the limiter middleware and error envelope become defaults in [`templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/); structured errors + request context feed [`blueprints/observability-stack/`](../../../blueprints/observability-stack/).
- 🏗️ **Capstone:** per-tenant limits wrap `capstone/app/` (checkpoint `ch26-enterprise-api`).